In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import Dict, List, Tuple

# ============================================================
# CONFIGURATION
# ============================================================

NUM_ROWS = 49000
SAMPLING_INTERVAL_MINUTES = 5
MISSION_START_DATE = "2026-01-01 00:00:00"
OUTPUT_FILENAME = "cubesat_telemetry.csv"

RANDOM_SEED = 42
TRAIN_FRACTION = 0.80
DEMO_ROWS = 2500
BALANCE_TOLERANCE = 0.03

ORBIT_ALTITUDE_KM = 550.0
ORBIT_PERIOD_MINUTES = 95.0
SUNLIGHT_FRACTION_MEAN = 0.60
ORBIT_DECAY_KM_TOTAL = 1.2

BATTERY_CAPACITY_AH_BOL = 12.0
BATTERY_V_MIN = 6.2
BATTERY_V_MAX = 8.4
BATTERY_OCV_MIN = 6.65
BATTERY_OCV_MAX = 8.32
BATTERY_R0_OHM = 0.070
BATTERY_SELF_DISCHARGE_PER_DAY = 0.00010
BATTERY_CHARGE_EFF_BOL = 0.985
BATTERY_DISCHARGE_EFF_BOL = 0.992

BUS_VOLTAGE_REG = 33.8
BUS_VOLTAGE_MIN = 20.0
BUS_VOLTAGE_MAX = 34.8
BUS_MAX_CURRENT = 15.0
BUS_REG_EFF = 0.955
MPPT_EFF = 0.965

SOLAR_VMPP_REF = 21.6
SOLAR_IMPP_REF = 6.2
SOLAR_LONG_TERM_DEGRADATION = 0.025
SOLAR_RADIATION_DEGRADATION = 0.020
SOLAR_VTEMP_COEFF = -0.0022
SOLAR_ITEMP_COEFF = 0.0005

BODY_TEMP_INIT = 16.0
BATTERY_TEMP_INIT = 15.0
CPU_TEMP_INIT = 42.0
WHEEL_TEMP_INIT = 24.0
TX_TEMP_INIT = 23.0

ADC_DROPOUT_RATE = 0.0010
ADC_SATURATION_RATE = 0.0008
SENSOR_LATENCY_PROB = 0.018
SENSOR_SPIKE_RATE = 0.0012
SENSOR_FREEZE_RATE = 0.0009
PACKET_MISSING_RATE = 0.0006

FAULT_NONE = 0
FAULT_POWER = 1
FAULT_BATTERY = 2
FAULT_SOLAR = 3
FAULT_WHEEL = 4
FAULT_THERMAL = 5
FAULT_COMM = 6
FAULT_SENSOR = 7

MODE_SAFE = 0
MODE_NOMINAL = 1
MODE_SCIENCE = 2
MODE_COMM = 3
MODE_DETUMBLE = 4
MODE_EMERGENCY = 5

FAULT_LABEL_NAMES = {
    FAULT_NONE: "NORMAL",
    FAULT_POWER: "POWER",
    FAULT_BATTERY: "BATTERY",
    FAULT_SOLAR: "SOLAR",
    FAULT_WHEEL: "REACTION_WHEEL",
    FAULT_THERMAL: "THERMAL",
    FAULT_COMM: "COMMUNICATION",
    FAULT_SENSOR: "SENSOR",
}

MODE_NAMES = {
    MODE_SAFE: "SAFE",
    MODE_NOMINAL: "NOMINAL",
    MODE_SCIENCE: "SCIENCE",
    MODE_COMM: "COMMUNICATION",
    MODE_DETUMBLE: "DETUMBLE",
    MODE_EMERGENCY: "EMERGENCY",
}

CLASS_TARGET_COUNTS = {
    FAULT_NONE: 7000,
    FAULT_POWER: 7000,
    FAULT_BATTERY: 7000,
    FAULT_SOLAR: 7000,
    FAULT_WHEEL: 7000,
    FAULT_THERMAL: 7000,
    FAULT_COMM: 7000,
    FAULT_SENSOR: 7000,
}

EPISODE_DURATION_RANGES = {
    FAULT_POWER: (38, 82),
    FAULT_BATTERY: (56, 120),
    FAULT_SOLAR: (44, 88),
    FAULT_WHEEL: (48, 96),
    FAULT_THERMAL: (52, 105),
    FAULT_COMM: (36, 76),
    FAULT_SENSOR: (26, 68),
}

EPISODE_MARGIN = 18
MISSION_IDLE_GAP = 10

np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

# ============================================================
# TIME BASE
# ============================================================

DT_HR = SAMPLING_INTERVAL_MINUTES / 60.0
DT_SEC = SAMPLING_INTERVAL_MINUTES * 60.0
t = np.arange(NUM_ROWS)
mission_days = t * SAMPLING_INTERVAL_MINUTES / (60.0 * 24.0)
mission_hours = t * SAMPLING_INTERVAL_MINUTES / 60.0

start_dt = datetime.strptime(MISSION_START_DATE, "%Y-%m-%d %H:%M:%S")
timestamps = [
    (start_dt + timedelta(minutes=int(i * SAMPLING_INTERVAL_MINUTES))).strftime("%Y-%m-%dT%H:%M:%SZ")
    for i in range(NUM_ROWS)
]

# ============================================================
# HELPERS
# ============================================================

def smooth_step(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)

def triangular_profile(length: int, grow_frac: float, peak_frac: float, rec_frac: float) -> Tuple[np.ndarray, np.ndarray]:
    x = np.linspace(0.0, 1.0, length)
    stage = np.zeros(length)
    phase = np.empty(length, dtype=object)

    a = max(0.05, grow_frac)
    b = min(0.95, a + peak_frac)
    c = min(0.99, b + rec_frac)

    for i, xi in enumerate(x):
        if xi < 0.08:
            phase[i] = "start"
            stage[i] = 0.06 + 0.50 * smooth_step(np.array([xi / 0.08]))[0]
        elif xi < a:
            phase[i] = "growth"
            u = (xi - 0.08) / max(a - 0.08, 1e-6)
            stage[i] = 0.18 + 0.52 * smooth_step(np.array([u]))[0]
        elif xi < b:
            phase[i] = "peak"
            u = (xi - a) / max(b - a, 1e-6)
            stage[i] = 0.70 + 0.30 * np.sin(np.pi * 0.5 * u)
        elif xi < c:
            phase[i] = "recovery"
            u = (xi - b) / max(c - b, 1e-6)
            stage[i] = 0.92 - 0.70 * smooth_step(np.array([u]))[0]
        else:
            phase[i] = "cooldown"
            u = (xi - c) / max(1.0 - c, 1e-6)
            stage[i] = 0.22 - 0.20 * smooth_step(np.array([u]))[0]

    stage += rng.normal(0.0, 0.015, length)
    return np.clip(stage, 0.0, 1.0), phase

@dataclass
class FaultEpisode:
    start: int
    end: int
    label: int
    episode_id: int
    severity_peak: float
    growth_frac: float
    peak_frac: float
    recovery_frac: float

# ============================================================
# ORBIT / SUNLIGHT
# ============================================================

orbit_phase = ((t * SAMPLING_INTERVAL_MINUTES) % ORBIT_PERIOD_MINUTES) / ORBIT_PERIOD_MINUTES
orbital_angle = 2.0 * np.pi * orbit_phase

beta_mod = 0.08 * np.sin(2.0 * np.pi * mission_days / 28.0 + 0.6)
sun_frac_dynamic = np.clip(SUNLIGHT_FRACTION_MEAN + beta_mod, 0.53, 0.67)
phase_centered = orbit_phase - sun_frac_dynamic
sun_transition = 0.5 * (1.0 - np.tanh(phase_centered / 0.040))
sunlight_binary = (sun_transition > 0.50).astype(int)

earth_ir_heat = 0.18 + 0.05 * np.sin(2.0 * np.pi * mission_days / 5.0 - 0.3)
earth_albedo_heat = 0.16 * sun_transition + 0.04 * np.sin(2.0 * np.pi * mission_days / 2.2 + 0.5) * sun_transition
space_env_base = -9.0 + 1.5 * np.sin(2.0 * np.pi * mission_days / 7.0)

orbital_decay = ORBIT_DECAY_KM_TOTAL * (mission_days / max(mission_days[-1], 1e-9))
altitude_true = (
    ORBIT_ALTITUDE_KM
    - orbital_decay
    + 0.65 * np.sin(2.0 * np.pi * mission_days / 14.0 + 0.4)
    + 0.28 * np.sin(2.0 * np.pi * mission_days / 5.5 - 0.6)
    + 0.08 * np.sin(2.0 * np.pi * orbit_phase + 0.2)
    + rng.normal(0.0, 0.03, NUM_ROWS)
)
altitude_true = np.clip(altitude_true, 545.0, 555.0)

# ============================================================
# MISSION WINDOWS
# ============================================================

comm_window = ((orbit_phase > 0.70) & (orbit_phase < 0.84)).astype(float)
science_window = ((orbit_phase > 0.10) & (orbit_phase < 0.36)).astype(float)
hk_window = ((orbit_phase > 0.50) & (orbit_phase < 0.58)).astype(float)
image_capture_window = ((orbit_phase > 0.18) & (orbit_phase < 0.24)).astype(float)
proc_window = (0.45 + 0.55 * np.sin(2.0 * np.pi * mission_days / 1.7 + 0.9)) * science_window
proc_window = np.clip(proc_window, 0.0, 1.0)

# ============================================================
# LONG-TERM AGEING
# ============================================================

age_frac = mission_days / max(mission_days[-1], 1e-9)

solar_eff_long = np.clip(1.0 - SOLAR_LONG_TERM_DEGRADATION * age_frac, 0.955, 1.0)
rad_deg = np.clip(1.0 - SOLAR_RADIATION_DEGRADATION * age_frac, 0.96, 1.0)

capacity_fade = 1.0 - 0.035 * age_frac
capacity_ah = BATTERY_CAPACITY_AH_BOL * capacity_fade
internal_res_growth = 1.0 + 0.32 * age_frac
charge_eff_aging = BATTERY_CHARGE_EFF_BOL * (1.0 - 0.035 * age_frac)
discharge_eff_aging = BATTERY_DISCHARGE_EFF_BOL * (1.0 - 0.025 * age_frac)

wheel_wear = 1.0 + 0.12 * age_frac
bearing_friction_scale = 1.0 + 0.18 * age_frac
control_eff = np.clip(1.0 - 0.06 * age_frac, 0.93, 1.0)
calibration_drift_scale = 1.0 + 0.08 * age_frac
sensor_drift_long = 0.20 * age_frac

# ============================================================
# BALANCED FAULT EPISODE PLANNING
# ============================================================

fault_label = np.zeros(NUM_ROWS, dtype=int)
fault_stage = np.zeros(NUM_ROWS, dtype=float)
fault_severity = np.zeros(NUM_ROWS, dtype=float)
fault_episode_id = np.zeros(NUM_ROWS, dtype=int)
fault_phase_code = np.zeros(NUM_ROWS, dtype=int)
occupied = np.zeros(NUM_ROWS, dtype=bool)
episodes: List[FaultEpisode] = []

phase_name_to_code = {"normal": 0, "start": 1, "growth": 2, "peak": 3, "recovery": 4, "cooldown": 5}

def get_fault_progression_params(label: int) -> Tuple[float, float, float]:
    if label == FAULT_POWER:
        return 0.36, 0.22, 0.24
    if label == FAULT_BATTERY:
        return 0.42, 0.18, 0.26
    if label == FAULT_SOLAR:
        return 0.32, 0.26, 0.24
    if label == FAULT_WHEEL:
        return 0.34, 0.24, 0.26
    if label == FAULT_THERMAL:
        return 0.44, 0.20, 0.24
    if label == FAULT_COMM:
        return 0.30, 0.22, 0.26
    if label == FAULT_SENSOR:
        return 0.20, 0.30, 0.24
    return 0.35, 0.20, 0.25

def plan_fault_episodes() -> None:
    global fault_label, fault_stage, fault_severity, fault_episode_id, fault_phase_code, occupied, episodes

    episode_counter = 1
    labels_to_plan = [FAULT_POWER, FAULT_BATTERY, FAULT_SOLAR, FAULT_WHEEL, FAULT_THERMAL, FAULT_COMM, FAULT_SENSOR]

    for label in labels_to_plan:
        desired = CLASS_TARGET_COUNTS[label]
        assigned = 0
        attempts = 0

        while assigned < desired and attempts < 50000:
            attempts += 1
            lo, hi = EPISODE_DURATION_RANGES[label]
            length = int(rng.integers(lo, hi + 1))
            start_idx = int(rng.integers(150, NUM_ROWS - length - 150))
            sl = slice(max(0, start_idx - EPISODE_MARGIN), min(NUM_ROWS, start_idx + length + EPISODE_MARGIN))

            if occupied[sl].any():
                continue

            occupied[start_idx:start_idx + length] = True
            growth_frac, peak_frac, recovery_frac = get_fault_progression_params(label)
            sev_peak = float(rng.uniform(72.0, 100.0))

            ep = FaultEpisode(
                start=start_idx,
                end=start_idx + length,
                label=label,
                episode_id=episode_counter,
                severity_peak=sev_peak,
                growth_frac=growth_frac,
                peak_frac=peak_frac,
                recovery_frac=recovery_frac,
            )
            episodes.append(ep)
            episode_counter += 1
            assigned += length

    for ep in episodes:
        L = ep.end - ep.start
        stage_local, phase_local = triangular_profile(L, ep.growth_frac, ep.peak_frac, ep.recovery_frac)
        sev_local = np.clip(ep.severity_peak * stage_local + rng.normal(0.0, 1.2, L), 0.0, 100.0)

        fault_label[ep.start:ep.end] = ep.label
        fault_stage[ep.start:ep.end] = np.maximum(fault_stage[ep.start:ep.end], stage_local)
        fault_severity[ep.start:ep.end] = np.maximum(fault_severity[ep.start:ep.end], sev_local)
        fault_episode_id[ep.start:ep.end] = ep.episode_id
        fault_phase_code[ep.start:ep.end] = np.array([phase_name_to_code[p] for p in phase_local])

plan_fault_episodes()

# ============================================================
# MODES
# ============================================================

mode = np.full(NUM_ROWS, MODE_NOMINAL, dtype=int)
mode[:18] = MODE_DETUMBLE
soc_proxy = 76.0 + 10.0 * np.sin(2.0 * np.pi * mission_days / 8.0) + 3.0 * sun_transition

for i in range(1, NUM_ROWS):
    if i < 18:
        mode[i] = MODE_DETUMBLE
        continue
    if fault_label[i] in (FAULT_POWER, FAULT_BATTERY) and fault_stage[i] > 0.72:
        mode[i] = MODE_EMERGENCY
    elif fault_label[i] in (FAULT_POWER, FAULT_BATTERY) and fault_stage[i] > 0.48:
        mode[i] = MODE_SAFE
    elif fault_label[i] == FAULT_WHEEL and fault_stage[i] > 0.55:
        mode[i] = MODE_DETUMBLE
    elif fault_label[i] == FAULT_THERMAL and fault_stage[i] > 0.55:
        mode[i] = MODE_SAFE
    elif soc_proxy[i] < 42.0 and sunlight_binary[i] == 0:
        mode[i] = MODE_SAFE
    elif comm_window[i] > 0.5:
        mode[i] = MODE_COMM
    elif science_window[i] > 0.5 and soc_proxy[i] > 48.0:
        mode[i] = MODE_SCIENCE
    else:
        mode[i] = MODE_NOMINAL

# ============================================================
# ADCS / WHEEL / GYRO TRUE DYNAMICS
# ============================================================

body_rate_true = np.zeros(NUM_ROWS)
body_rate_true[0] = 0.85
stored_momentum = np.zeros(NUM_ROWS)
wheel_rpm_true = np.zeros(NUM_ROWS)
desat_flag = np.zeros(NUM_ROWS)

for i in range(1, NUM_ROWS):
    env_torque = (
        0.008
        + 0.012 * np.abs(np.sin(2.0 * np.pi * mission_days[i] / 1.9 + 0.7))
        + 0.010 * np.abs(np.sin(orbital_angle[i] + 0.3))
        + 0.003 * (1.0 - sunlight_binary[i])
    )

    micro_slew = 0.0
    if mode[i] == MODE_SCIENCE and proc_window[i] > 0.25:
        micro_slew += 0.05 * np.abs(np.sin(2.0 * np.pi * mission_days[i] / 0.42))
    if mode[i] == MODE_COMM and comm_window[i] > 0.6:
        micro_slew += 0.03 * np.abs(np.sin(2.0 * np.pi * mission_days[i] / 0.31 + 0.9))
    if image_capture_window[i] > 0.5:
        micro_slew += 0.025

    if mode[i] == MODE_DETUMBLE:
        ctrl = 0.095 * control_eff[i]
        rate_target = 0.020
    elif mode[i] == MODE_SCIENCE:
        ctrl = 0.062 * control_eff[i]
        rate_target = 0.028 + micro_slew
    elif mode[i] == MODE_COMM:
        ctrl = 0.055 * control_eff[i]
        rate_target = 0.035 + micro_slew
    elif mode[i] == MODE_SAFE:
        ctrl = 0.032 * control_eff[i]
        rate_target = 0.080
    elif mode[i] == MODE_EMERGENCY:
        ctrl = 0.025 * control_eff[i]
        rate_target = 0.120
    else:
        ctrl = 0.043 * control_eff[i]
        rate_target = 0.045 + micro_slew

    if fault_label[i] == FAULT_WHEEL:
        ctrl *= (1.0 - 0.44 * fault_stage[i])
        env_torque += 0.05 * fault_stage[i]

    rate_err = body_rate_true[i - 1] - rate_target
    rate_dot = 0.36 * env_torque - ctrl * rate_err + rng.normal(0.0, 0.0015)
    if mode[i] == MODE_DETUMBLE:
        rate_dot -= 0.045 * np.sign(body_rate_true[i - 1])

    body_rate_true[i] = body_rate_true[i - 1] + DT_HR * 12.0 * rate_dot
    body_rate_true[i] = np.clip(body_rate_true[i], 0.005, 2.8)

    stored_momentum[i] = stored_momentum[i - 1] + DT_HR * (
        0.95 * np.abs(rate_err) + 0.25 * env_torque + 0.16 * micro_slew
    )

    desat_threshold = 1.10 * (1.0 - 0.05 * age_frac[i])
    if stored_momentum[i] > desat_threshold or (mode[i] == MODE_DETUMBLE and stored_momentum[i] > 0.55):
        desat_flag[i] = 1.0
        stored_momentum[i] *= 0.52

    friction_term = 34.0 * bearing_friction_scale[i]
    base_rpm_cmd = (
        240.0 * np.sin(2.0 * np.pi * mission_days[i] / 0.79 + 0.5)
        + 180.0 * np.sin(2.0 * np.pi * mission_days[i] / 2.4 + 1.3)
        + 780.0 * (body_rate_true[i] - rate_target)
        + 620.0 * stored_momentum[i]
    )

    if mode[i] == MODE_SCIENCE:
        base_rpm_cmd += 190.0
    elif mode[i] == MODE_COMM:
        base_rpm_cmd += 110.0
    elif mode[i] == MODE_SAFE:
        base_rpm_cmd *= 0.72
    elif mode[i] == MODE_EMERGENCY:
        base_rpm_cmd *= 0.64
    elif mode[i] == MODE_DETUMBLE:
        base_rpm_cmd += 520.0 * np.sign(np.sin(2.0 * np.pi * mission_days[i] / 0.22))

    if desat_flag[i] > 0.5:
        base_rpm_cmd *= 0.40

    if fault_label[i] == FAULT_WHEEL:
        base_rpm_cmd += (950.0 + 1250.0 * fault_stage[i]) * np.sin(2.0 * np.pi * mission_days[i] / 0.12 + 1.7)

    rpm_dot = 0.17 * (base_rpm_cmd - wheel_rpm_true[i - 1]) - friction_term * np.sign(wheel_rpm_true[i - 1]) / 120.0
    wheel_rpm_true[i] = wheel_rpm_true[i - 1] + rpm_dot + rng.normal(0.0, 7.0 * wheel_wear[i])

wheel_rpm_true = np.clip(wheel_rpm_true, -3500.0, 3500.0)

# ============================================================
# CPU TRUE WORKLOAD
# ============================================================

cpu_target = np.zeros(NUM_ROWS)
for i in range(NUM_ROWS):
    if mode[i] == MODE_SAFE:
        target = 18.0 + 4.0 * hk_window[i]
    elif mode[i] == MODE_NOMINAL:
        target = 28.0 + 8.0 * hk_window[i] + 5.0 * proc_window[i]
    elif mode[i] == MODE_SCIENCE:
        target = 46.0 + 16.0 * proc_window[i] + 9.0 * image_capture_window[i] + 7.0 * hk_window[i]
    elif mode[i] == MODE_COMM:
        target = 40.0 + 12.0 * comm_window[i] + 5.0 * hk_window[i]
    elif mode[i] == MODE_EMERGENCY:
        target = 58.0 + 10.0 * hk_window[i]
    else:
        target = 55.0 + 8.0 * hk_window[i]
    target += 3.0 * np.sin(2.0 * np.pi * mission_days[i] / 1.3 + 0.5)
    target += 2.0 * np.sin(2.0 * np.pi * mission_days[i] / 5.6 - 0.8)
    cpu_target[i] = target

cpu_usage_true = np.zeros(NUM_ROWS)
cpu_usage_true[0] = 30.0
for i in range(1, NUM_ROWS):
    tau = 0.18 if mode[i] in (MODE_SCIENCE, MODE_COMM, MODE_DETUMBLE, MODE_EMERGENCY) else 0.12
    cpu_usage_true[i] = cpu_usage_true[i - 1] + tau * (cpu_target[i] - cpu_usage_true[i - 1])

cpu_usage_true[fault_label == FAULT_THERMAL] += 9.0 + 26.0 * fault_stage[fault_label == FAULT_THERMAL]
cpu_usage_true = np.clip(cpu_usage_true, 15.0, 100.0)

# ============================================================
# THERMAL TRUE MODEL
# ============================================================

body_temp_true = np.zeros(NUM_ROWS)
battery_temp_true = np.zeros(NUM_ROWS)
cpu_temp_true = np.zeros(NUM_ROWS)
wheel_temp_true = np.zeros(NUM_ROWS)
tx_temp_true = np.zeros(NUM_ROWS)
panel_temp_true = np.zeros(NUM_ROWS)

body_temp_true[0] = BODY_TEMP_INIT
battery_temp_true[0] = BATTERY_TEMP_INIT
cpu_temp_true[0] = CPU_TEMP_INIT
wheel_temp_true[0] = WHEEL_TEMP_INIT
tx_temp_true[0] = TX_TEMP_INIT
panel_temp_true[0] = BODY_TEMP_INIT + 5.0

batt_current_proxy = np.zeros(NUM_ROWS)
for i in range(NUM_ROWS):
    base_load = {
        MODE_SAFE: 8.2,
        MODE_NOMINAL: 9.4,
        MODE_SCIENCE: 10.9,
        MODE_COMM: 11.0,
        MODE_DETUMBLE: 12.0,
        MODE_EMERGENCY: 12.3,
    }[mode[i]]
    cpu_load = 0.040 * np.maximum(cpu_usage_true[i] - 20.0, 0.0)
    rw_load = 0.00085 * np.abs(wheel_rpm_true[i]) + 0.30 * desat_flag[i]
    tx_load = 1.10 * comm_window[i] if mode[i] == MODE_COMM else 0.20 * comm_window[i]
    payload_load = 0.65 * science_window[i] + 0.35 * image_capture_window[i] + 0.40 * proc_window[i]
    hk_load = 0.15 * hk_window[i]
    load_current_proxy = np.clip(base_load + cpu_load + rw_load + tx_load + payload_load + hk_load, 8.0, BUS_MAX_CURRENT)
    solar_i_proxy = np.clip(SOLAR_IMPP_REF * sun_transition[i] * solar_eff_long[i] * rad_deg[i], 0.0, 7.0)
    batt_current_proxy[i] = np.maximum(0.0, load_current_proxy - 0.52 * solar_i_proxy)

for i in range(1, NUM_ROWS):
    incidence_raw = (0.55 + 0.45 * np.sin(np.pi * np.clip(orbit_phase[i] / np.maximum(sun_frac_dynamic[i], 1e-6), 0.0, 1.0))) * sun_transition[i]
    incidence_raw += 0.12 * np.sin(2.0 * np.pi * mission_days[i] / 3.9 + 0.7) * sun_transition[i]
    incidence_raw = np.clip(incidence_raw, 0.0, 1.0)

    panel_eq = (
        body_temp_true[i - 1]
        + 12.0 * incidence_raw
        + 6.0 * sun_transition[i]
        + 1.8 * earth_albedo_heat[i]
        + 1.5 * earth_ir_heat[i]
    )
    if fault_label[i] == FAULT_SOLAR:
        panel_eq += 2.0 * fault_stage[i]
    panel_temp_true[i] = panel_temp_true[i - 1] + 0.095 * (panel_eq - panel_temp_true[i - 1]) + rng.normal(0.0, 0.03)

    body_eq = (
        8.0 + space_env_base[i]
        + 0.32 * panel_temp_true[i]
        + 1.8 * earth_ir_heat[i]
        + 1.2 * earth_albedo_heat[i]
        + 0.10 * cpu_usage_true[i]
        + 0.0055 * np.abs(wheel_rpm_true[i])
        + 0.08 * tx_temp_true[i - 1]
    )
    if fault_label[i] == FAULT_WHEEL:
        body_eq += 3.0 * fault_stage[i]
    if fault_label[i] == FAULT_THERMAL:
        body_eq += 2.4 * fault_stage[i]
    body_temp_true[i] = body_temp_true[i - 1] + 0.055 * (body_eq - body_temp_true[i - 1]) + rng.normal(0.0, 0.025)

    cpu_eq = body_temp_true[i] + 18.0 + 0.30 * cpu_usage_true[i] + 0.10 * comm_window[i] * 12.0
    if fault_label[i] == FAULT_THERMAL:
        cpu_eq += 13.5 * fault_stage[i]
    cpu_temp_true[i] = cpu_temp_true[i - 1] + 0.085 * (cpu_eq - cpu_temp_true[i - 1]) + rng.normal(0.0, 0.035)

    if cpu_temp_true[i] > 78.0:
        throttle = np.clip((cpu_temp_true[i] - 78.0) / 10.0, 0.0, 0.35)
        cpu_usage_true[i] *= (1.0 - throttle)

    wheel_eq = body_temp_true[i] + 6.0 + 0.0105 * np.abs(wheel_rpm_true[i]) + 0.8 * bearing_friction_scale[i]
    if fault_label[i] == FAULT_WHEEL:
        wheel_eq += 16.0 * fault_stage[i]
    wheel_temp_true[i] = wheel_temp_true[i - 1] + 0.080 * (wheel_eq - wheel_temp_true[i - 1]) + rng.normal(0.0, 0.035)

    tx_eq = body_temp_true[i] + 4.0 + 5.0 * comm_window[i] + 0.12 * np.maximum(cpu_usage_true[i] - 25.0, 0.0)
    if fault_label[i] == FAULT_COMM:
        tx_eq += 4.5 * fault_stage[i]
    tx_temp_true[i] = tx_temp_true[i - 1] + 0.070 * (tx_eq - tx_temp_true[i - 1]) + rng.normal(0.0, 0.030)

    batt_heat = 0.26 * (np.abs(batt_current_proxy[i]) ** 1.15) * internal_res_growth[i]
    batt_eq = (
        body_temp_true[i]
        + 1.6
        + 0.82 * batt_heat
        + 0.018 * np.maximum(panel_temp_true[i] - body_temp_true[i], 0.0)
        + 0.030 * np.maximum(cpu_temp_true[i] - body_temp_true[i], 0.0)
        + 0.015 * np.maximum(np.abs(wheel_rpm_true[i]) - 600.0, 0.0) / 100.0
    )
    if fault_label[i] == FAULT_BATTERY:
        batt_eq += 8.5 * fault_stage[i]
    if fault_label[i] == FAULT_THERMAL:
        batt_eq += 3.0 * fault_stage[i]
    battery_temp_true[i] = battery_temp_true[i - 1] + 0.045 * (batt_eq - battery_temp_true[i - 1]) + rng.normal(0.0, 0.025)

panel_temp_true = np.clip(panel_temp_true, -20.0, 70.0)
body_temp_true = np.clip(body_temp_true, -5.0, 50.0)
battery_temp_true = np.clip(battery_temp_true, -15.0, 55.0)
cpu_temp_true = np.clip(cpu_temp_true, 35.0, 85.0)
wheel_temp_true = np.clip(wheel_temp_true, 20.0, 65.0)
tx_temp_true = np.clip(tx_temp_true, 10.0, 70.0)

# ============================================================
# SOLAR TRUE ELECTRICAL MODEL
# ============================================================

solar_voltage_true = np.zeros(NUM_ROWS)
solar_current_true = np.zeros(NUM_ROWS)
solar_power_true = np.zeros(NUM_ROWS)
sun_incidence = np.zeros(NUM_ROWS)

for i in range(NUM_ROWS):
    sun_orbit_frac = np.clip(orbit_phase[i] / np.maximum(sun_frac_dynamic[i], 1e-6), 0.0, 1.0)
    incidence = (0.55 + 0.45 * np.sin(np.pi * sun_orbit_frac)) * sun_transition[i]
    incidence += 0.12 * np.sin(2.0 * np.pi * mission_days[i] / 3.9 + 0.7) * sun_transition[i]
    incidence = np.clip(incidence, 0.0, 1.0)
    sun_incidence[i] = incidence

    array_temp = panel_temp_true[i]
    current_temp_factor = 1.0 + SOLAR_ITEMP_COEFF * (array_temp - 25.0)
    voltage_temp_factor = 1.0 + SOLAR_VTEMP_COEFF * (array_temp - 25.0)

    imp = SOLAR_IMPP_REF * incidence * solar_eff_long[i] * rad_deg[i] * (0.975 - 0.010 * age_frac[i]) * current_temp_factor
    vmpp = SOLAR_VMPP_REF * voltage_temp_factor * (0.996 - 0.006 * age_frac[i])

    if fault_label[i] == FAULT_SOLAR:
        imp *= np.clip(1.0 - 0.90 * fault_stage[i], 0.03, 1.0)
        vmpp *= np.clip(1.0 - 0.20 * fault_stage[i], 0.72, 1.0)

    if sunlight_binary[i] == 1 or sun_transition[i] > 0.03:
        solar_voltage_true[i] = np.clip(vmpp + rng.normal(0.0, 0.05), 18.0, 24.0)
        solar_current_true[i] = np.clip(imp + rng.normal(0.0, 0.05), 0.0, 7.0)
    else:
        solar_voltage_true[i] = np.clip(0.22 * sun_transition[i] + rng.normal(0.0, 0.01), 0.0, 0.8)
        solar_current_true[i] = np.clip(0.04 * sun_transition[i] + rng.normal(0.0, 0.01), 0.0, 0.20)

    solar_power_true[i] = solar_voltage_true[i] * solar_current_true[i] * MPPT_EFF

# ============================================================
# BATTERY / EPS TRUE MODEL
# ============================================================

def battery_ocv_from_soc_temp(soc, temp_c, age_local=0.0):
    soc = np.clip(np.asarray(soc, dtype=float), 0.0, 100.0)
    temp_c = np.asarray(temp_c, dtype=float)
    age_local = np.asarray(age_local, dtype=float)
    soc_n = soc / 100.0
    ocv = (
        6.42
        + 1.62 * soc_n
        + 0.18 * np.tanh((soc_n - 0.55) / 0.16)
        - 0.035 * np.exp(-7.5 * soc_n)
        - 0.0018 * np.maximum(0.0, 12.0 - temp_c)
        + 0.0007 * np.maximum(temp_c - 28.0, 0.0)
        - 0.025 * age_local
    )
    return np.clip(ocv, BATTERY_OCV_MIN, BATTERY_OCV_MAX)

def battery_capacity_ah_effective(capacity_bol_ah, age_local, stage_local, label_local):
    calendar_fade = 0.018 * age_local
    cycle_fade = 0.020 * age_local
    fade = calendar_fade + cycle_fade
    cap = capacity_bol_ah * (1.0 - fade)
    batt_fault_mask = (label_local == FAULT_BATTERY).astype(float)
    cap *= (1.0 - batt_fault_mask * (0.04 + 0.20 * stage_local))
    return np.clip(cap, 0.70 * capacity_bol_ah, capacity_bol_ah)

def battery_internal_resistance_ohm(r0_ohm, temp_c, age_local, stage_local, label_local):
    temp_c = np.asarray(temp_c, dtype=float)
    age_local = np.asarray(age_local, dtype=float)
    temp_factor = 1.0 + 0.0045 * np.maximum(0.0, 18.0 - temp_c) + 0.0015 * np.maximum(temp_c - 32.0, 0.0)
    age_factor = 1.0 + 0.28 * age_local
    r = r0_ohm * temp_factor * age_factor
    batt_fault_mask = (label_local == FAULT_BATTERY).astype(float)
    r *= (1.0 + batt_fault_mask * (0.20 + 0.95 * stage_local))
    return np.clip(r, 0.04, 0.24)

def battery_charge_discharge_efficiency(temp_c, age_local, stage_local, label_local):
    temp_c = np.asarray(temp_c, dtype=float)
    age_local = np.asarray(age_local, dtype=float)
    charge_eff = (
        BATTERY_CHARGE_EFF_BOL
        * (1.0 - 0.030 * age_local)
        * np.clip(1.0 - 0.0035 * np.abs(temp_c - 24.0), 0.82, 1.0)
    )
    discharge_eff = (
        BATTERY_DISCHARGE_EFF_BOL
        * (1.0 - 0.020 * age_local)
        * np.clip(1.0 - 0.0020 * np.abs(temp_c - 24.0), 0.86, 1.0)
    )
    batt_fault_mask = (label_local == FAULT_BATTERY).astype(float)
    charge_eff *= (1.0 - batt_fault_mask * (0.05 + 0.24 * stage_local))
    discharge_eff *= (1.0 - batt_fault_mask * (0.03 + 0.10 * stage_local))
    return np.clip(charge_eff, 0.70, 0.995), np.clip(discharge_eff, 0.78, 0.998)

def battery_rate_limits_amps(capacity_ah_eff, temp_c, age_local, stage_local, label_local):
    temp_c = np.asarray(temp_c, dtype=float)
    age_local = np.asarray(age_local, dtype=float)
    charge_c_nom = 0.32
    discharge_c_nom = 0.58
    temp_charge_factor = np.clip(
        1.0 - 0.012 * np.maximum(0.0, 8.0 - temp_c) - 0.006 * np.maximum(temp_c - 35.0, 0.0),
        0.45, 1.0
    )
    temp_discharge_factor = np.clip(
        1.0 - 0.008 * np.maximum(0.0, 0.0 - temp_c) - 0.004 * np.maximum(temp_c - 40.0, 0.0),
        0.50, 1.0
    )
    age_factor_charge = np.clip(1.0 - 0.10 * age_local, 0.82, 1.0)
    age_factor_discharge = np.clip(1.0 - 0.08 * age_local, 0.85, 1.0)

    max_charge_a = capacity_ah_eff * charge_c_nom * temp_charge_factor * age_factor_charge
    max_discharge_a = capacity_ah_eff * discharge_c_nom * temp_discharge_factor * age_factor_discharge

    batt_fault_mask = (label_local == FAULT_BATTERY).astype(float)
    max_charge_a *= (1.0 - batt_fault_mask * (0.08 + 0.35 * stage_local))
    max_discharge_a *= (1.0 - batt_fault_mask * (0.04 + 0.15 * stage_local))
    return np.clip(max_charge_a, 0.6, 5.0), np.clip(max_discharge_a, 1.2, 8.0)

bus_current_true = np.zeros(NUM_ROWS)
bus_voltage_true = np.zeros(NUM_ROWS)
battery_soc_true = np.zeros(NUM_ROWS)
battery_voltage_true = np.zeros(NUM_ROWS)
battery_current_true = np.zeros(NUM_ROWS)
battery_charge_ah = np.zeros(NUM_ROWS)
load_power_true = np.zeros(NUM_ROWS)
mppt_output_power_true = np.zeros(NUM_ROWS)
bus_power_true = np.zeros(NUM_ROWS)

battery_soc_true[0] = 78.0
cap_eff_0 = battery_capacity_ah_effective(BATTERY_CAPACITY_AH_BOL, age_frac[0], fault_stage[0], fault_label[0])
battery_charge_ah[0] = cap_eff_0 * battery_soc_true[0] / 100.0
battery_voltage_true[0] = battery_ocv_from_soc_temp(battery_soc_true[0], battery_temp_true[0], age_frac[0])
bus_voltage_true[0] = BUS_VOLTAGE_REG - 0.10

for i in range(NUM_ROWS):
    base_load = {
        MODE_SAFE: 8.2,
        MODE_NOMINAL: 9.4,
        MODE_SCIENCE: 10.8,
        MODE_COMM: 11.0,
        MODE_DETUMBLE: 12.0,
        MODE_EMERGENCY: 12.4,
    }[mode[i]]

    cpu_load = 0.040 * max(cpu_usage_true[i] - 20.0, 0.0)
    rw_load = 0.00085 * abs(wheel_rpm_true[i]) + 0.30 * desat_flag[i]
    tx_load = 1.10 * comm_window[i] if mode[i] == MODE_COMM else 0.20 * comm_window[i]
    hk_load = 0.15 * hk_window[i]
    payload_load = 0.65 * science_window[i] + 0.35 * image_capture_window[i] + 0.40 * proc_window[i]

    bus_current = base_load + cpu_load + rw_load + tx_load + hk_load + payload_load

    if fault_label[i] == FAULT_POWER:
        bus_current -= 0.12 + 0.85 * fault_stage[i]
    if fault_label[i] == FAULT_WHEEL:
        bus_current += 0.22 + 1.35 * fault_stage[i]
    if fault_label[i] == FAULT_THERMAL:
        bus_current += 0.16 + 0.90 * fault_stage[i]
    if fault_label[i] == FAULT_BATTERY:
        bus_current -= 0.10 + 0.50 * fault_stage[i]
    if fault_label[i] == FAULT_COMM:
        bus_current += 0.10 * fault_stage[i]

    bus_current = np.clip(bus_current, 7.6, BUS_MAX_CURRENT)
    bus_current_true[i] = bus_current

    cap_eff = battery_capacity_ah_effective(BATTERY_CAPACITY_AH_BOL, age_frac[i], fault_stage[i], fault_label[i])

    if i == 0:
        charge_prev = battery_charge_ah[0]
        soc_prev = battery_soc_true[0]
    else:
        charge_prev = battery_charge_ah[i - 1]
        soc_prev = battery_soc_true[i - 1]

    soc_prev = np.clip(soc_prev, 18.0, 96.0)
    ocv_prev = battery_ocv_from_soc_temp(soc_prev, battery_temp_true[i], age_frac[i])
    r_int = battery_internal_resistance_ohm(BATTERY_R0_OHM, battery_temp_true[i], age_frac[i], fault_stage[i], fault_label[i])
    charge_eff, discharge_eff = battery_charge_discharge_efficiency(battery_temp_true[i], age_frac[i], fault_stage[i], fault_label[i])
    max_charge_a, max_discharge_a = battery_rate_limits_amps(cap_eff, battery_temp_true[i], age_frac[i], fault_stage[i], fault_label[i])

    solar_charge_power = solar_power_true[i] * np.clip(0.90 + 0.08 * sun_transition[i], 0.88, 0.98)
    load_power = (BUS_VOLTAGE_REG * bus_current / BUS_REG_EFF) * (
        0.985 + 0.015 * np.sin(2.0 * np.pi * mission_days[i] / 0.9 + 0.4)
    )

    mppt_output_power_true[i] = solar_charge_power
    load_power_true[i] = load_power
    net_power = load_power - solar_charge_power

    sunlight_boost_w = 8.0 * np.clip(sun_transition[i] - 0.25, 0.0, 1.0)
    eclipse_drag_w = 5.0 * np.clip(0.30 - sun_transition[i], 0.0, 0.30) / 0.30
    soc_high_taper = np.clip((90.0 - soc_prev) / 18.0, 0.15, 1.0)
    soc_low_guard = np.clip((soc_prev - 38.0) / 14.0, 0.20, 1.0)

    net_power -= sunlight_boost_w * soc_high_taper
    net_power += eclipse_drag_w / soc_low_guard

    if fault_label[i] == FAULT_SOLAR and sunlight_binary[i] == 1:
        net_power += 5.0 * fault_stage[i]
    if fault_label[i] == FAULT_BATTERY:
        net_power += 2.5 * fault_stage[i]
    if fault_label[i] == FAULT_POWER:
        net_power += 6.0 * fault_stage[i]

    desired_batt_power = net_power

    if desired_batt_power >= 0.0:
        desired_discharge_a = desired_batt_power / max(ocv_prev + 0.08, 6.5)
        batt_current = np.clip(desired_discharge_a, 0.0, max_discharge_a)
        battery_support_power = batt_current * max(ocv_prev - 0.5 * batt_current * r_int, 6.2)
        power_deficit = max(0.0, desired_batt_power - battery_support_power)
    else:
        desired_charge_a = (-desired_batt_power) / max(ocv_prev + 0.20, 6.6)
        charge_acceptance = np.clip((92.0 - soc_prev) / 22.0, 0.12, 1.0)
        batt_current = -np.clip(desired_charge_a * charge_acceptance, 0.0, max_charge_a)
        battery_support_power = batt_current * max(ocv_prev, 6.2)
        power_deficit = 0.0

    self_discharge_ah = cap_eff * BATTERY_SELF_DISCHARGE_PER_DAY * DT_HR / 24.0

    if batt_current < 0.0:
        delta_ah = (-batt_current) * charge_eff * DT_HR
    else:
        delta_ah = -(batt_current / max(discharge_eff, 1e-6)) * DT_HR

    if i > 0:
        prev_current = battery_current_true[i - 1]
        relax = np.exp(-DT_SEC / 900.0)
        polarization_memory = 0.010 * prev_current * relax
    else:
        polarization_memory = 0.0

    charge_new = charge_prev + delta_ah - self_discharge_ah + rng.normal(0.0, 0.00035)

    if fault_label[i] == FAULT_BATTERY:
        charge_new -= (0.010 + 0.070 * fault_stage[i]) * DT_HR
    if fault_label[i] == FAULT_POWER:
        charge_new -= (0.004 + 0.018 * fault_stage[i]) * DT_HR

    charge_floor = 0.18 * cap_eff
    charge_ceiling = 0.96 * cap_eff
    charge_new = np.clip(charge_new, charge_floor, charge_ceiling)

    soc_now = 100.0 * charge_new / max(cap_eff, 1e-6)
    soc_now = np.clip(soc_now, 18.0, 96.0)

    if i > 0:
        max_soc_step = 0.85 if fault_label[i] != FAULT_BATTERY else 1.35
        soc_now = np.clip(soc_now, battery_soc_true[i - 1] - max_soc_step, battery_soc_true[i - 1] + max_soc_step)
        charge_new = soc_now / 100.0 * cap_eff

    ocv_now = battery_ocv_from_soc_temp(soc_now, battery_temp_true[i], age_frac[i])

    temp_voltage_term = (
        -0.0019 * max(0.0, 12.0 - battery_temp_true[i])
        + 0.0006 * max(battery_temp_true[i] - 30.0, 0.0)
    )
    age_voltage_term = -0.030 * age_frac[i]

    if batt_current >= 0.0:
        ir_drop = batt_current * r_int
    else:
        ir_drop = batt_current * (0.72 * r_int)

    voltage_recovery = 0.0
    if i > 0:
        prev_i = battery_current_true[i - 1]
        if prev_i > 0.5 and batt_current < 0.2:
            voltage_recovery = 0.020 + 0.012 * min(prev_i, 3.0) / 3.0
        elif prev_i < -0.4 and batt_current > -0.1:
            voltage_recovery = -0.008

    terminal_v = ocv_now - ir_drop + temp_voltage_term + age_voltage_term + voltage_recovery - polarization_memory

    if fault_label[i] == FAULT_BATTERY:
        terminal_v -= 0.04 + 0.26 * fault_stage[i]
    if fault_label[i] == FAULT_POWER:
        terminal_v -= 0.02 + 0.08 * fault_stage[i]

    terminal_v = np.clip(terminal_v, BATTERY_V_MIN, BATTERY_V_MAX)

    battery_charge_ah[i] = charge_new
    battery_soc_true[i] = soc_now
    battery_current_true[i] = batt_current
    battery_voltage_true[i] = terminal_v

    solar_headroom = np.clip((solar_charge_power - load_power) / 70.0, -1.0, 1.0)
    batt_support = np.clip(batt_current / 4.5, -1.0, 1.0)
    load_stress = np.clip((bus_current - 10.5) / 4.0, 0.0, 1.2)
    low_batt_stress = np.clip((44.0 - soc_now) / 20.0, 0.0, 1.2)

    base_vreg = (
        BUS_VOLTAGE_REG
        - 0.030 * (bus_current - 10.0)
        + 0.090 * solar_headroom
        - 0.060 * np.maximum(batt_support, 0.0)
        + 0.040 * np.maximum(-batt_support, 0.0)
        - 0.110 * low_batt_stress
        + 0.018 * np.sin(2.0 * np.pi * mission_days[i] / 0.75 + 0.3)
        + 0.012 * np.sin(2.0 * np.pi * orbit_phase[i] + 0.8)
    )

    if terminal_v < 7.0 and sunlight_binary[i] == 0:
        base_vreg -= 0.20 * (7.0 - terminal_v)

    if fault_label[i] == FAULT_POWER:
        base_vreg -= 0.34 + 2.00 * fault_stage[i] + 1.45 * max(fault_stage[i] - 0.72, 0.0)
    if fault_label[i] == FAULT_SOLAR and sunlight_binary[i] == 1:
        base_vreg -= 0.14 + 0.58 * fault_stage[i]
    if fault_label[i] == FAULT_BATTERY:
        base_vreg -= 0.08 + 0.24 * fault_stage[i]

    if power_deficit > 0.0:
        base_vreg -= min(2.8, 0.060 * power_deficit + 0.18 * load_stress)

    target_v = np.clip(base_vreg, BUS_VOLTAGE_MIN, BUS_VOLTAGE_MAX)

    if i == 0:
        bus_v_now = target_v
    else:
        prev_bus_v = bus_voltage_true[i - 1]
        tau = 0.24 if target_v > prev_bus_v else 0.18
        bus_v_now = prev_bus_v + tau * (target_v - prev_bus_v)
        max_step = 0.18 if fault_label[i] != FAULT_POWER else 0.38
        bus_v_now = np.clip(bus_v_now, prev_bus_v - max_step, prev_bus_v + max_step)

    bus_voltage_true[i] = np.clip(bus_v_now, BUS_VOLTAGE_MIN, BUS_VOLTAGE_MAX)
    bus_power_true[i] = bus_voltage_true[i] * bus_current_true[i]

for i in range(1, NUM_ROWS):
    if fault_label[i] == FAULT_WHEEL and fault_stage[i] > 0.60:
        mode[i] = MODE_DETUMBLE
    elif fault_label[i] in (FAULT_POWER, FAULT_BATTERY) and fault_stage[i] > 0.72:
        mode[i] = MODE_EMERGENCY
    elif fault_label[i] in (FAULT_POWER, FAULT_BATTERY) and fault_stage[i] > 0.50:
        mode[i] = MODE_SAFE
    elif battery_soc_true[i] < 35.0 and sunlight_binary[i] == 0:
        mode[i] = MODE_SAFE
    elif comm_window[i] > 0.5:
        mode[i] = MODE_COMM
    elif science_window[i] > 0.5 and battery_soc_true[i] > 45.0:
        mode[i] = MODE_SCIENCE
    elif mode[i] != MODE_DETUMBLE:
        mode[i] = MODE_NOMINAL

# ============================================================
# COMMUNICATION TRUE MODEL
# ============================================================

signal_strength_true = np.zeros(NUM_ROWS)
for i in range(NUM_ROWS):
    elev_arg = np.clip((orbit_phase[i] - 0.68) / 0.18, 0.0, 1.0)
    elevation_proxy = np.clip(np.sin(np.pi * elev_arg), 0.0, 1.0)
    wheel_jitter = np.std(wheel_rpm_true[max(0, i - 6):i + 1]) / 1000.0 if i > 2 else 0.05

    antenna_pointing = np.clip(
        0.93 - 0.18 * body_rate_true[i] - 0.14 * wheel_jitter + 0.08 * comm_window[i] - 0.04 * np.maximum(0.0, 7.0 - battery_voltage_true[i]),
        0.10, 1.0
    )
    if mode[i] == MODE_COMM:
        antenna_pointing = np.clip(antenna_pointing + 0.08, 0.10, 1.0)
    if mode[i] == MODE_DETUMBLE:
        antenna_pointing = np.clip(antenna_pointing - 0.18, 0.05, 1.0)

    tx_power_factor = np.clip(1.0 - 0.05 * np.maximum(0.0, 7.0 - battery_voltage_true[i]), 0.75, 1.0)
    temp_penalty = np.clip(1.0 - 0.004 * np.maximum(0.0, tx_temp_true[i] - 45.0), 0.80, 1.0)

    link_margin = (
        -104.0
        + 20.0 * elevation_proxy
        + 5.5 * antenna_pointing
        + 3.0 * comm_window[i]
        + 2.0 * tx_power_factor
        + 1.2 * temp_penalty
        - 0.8 * body_rate_true[i]
    )

    if fault_label[i] == FAULT_COMM:
        link_margin -= 20.0 * fault_stage[i] + 4.0
    if fault_label[i] == FAULT_POWER:
        link_margin -= 2.5 * fault_stage[i]

    signal_strength_true[i] = np.clip(link_margin + rng.normal(0.0, 0.35), -120.0, -70.0)

# ============================================================
# SENSOR MODEL
# ============================================================

def lowpass_latency(sig: np.ndarray, hold_prob: float = SENSOR_LATENCY_PROB) -> np.ndarray:
    out = sig.copy()
    for i in range(1, len(out)):
        if rng.random() < hold_prob:
            out[i] = out[i - 1]
        else:
            out[i] = 0.78 * out[i] + 0.22 * out[i - 1]
    return out

def add_sensor_effects(
    sig: np.ndarray,
    noise_std: float,
    bias0: float,
    drift_span: float,
    qstep: float = None,
    lo: float = None,
    hi: float = None,
    dropout_rate: float = ADC_DROPOUT_RATE,
    sat_rate: float = ADC_SATURATION_RATE,
    spike_rate: float = SENSOR_SPIKE_RATE,
    freeze_rate: float = SENSOR_FREEZE_RATE,
    freeze_len_range: Tuple[int, int] = (2, 8),
    sensor_fault_mask: np.ndarray = None,
    sensor_fault_strength: np.ndarray = None,
) -> np.ndarray:
    n = len(sig)
    drift = np.linspace(0.0, drift_span, n) + 0.15 * drift_span * np.sin(2.0 * np.pi * np.arange(n) / max(n / 3.0, 1.0))
    y = sig + bias0 + drift + rng.normal(0.0, noise_std, n)
    y = lowpass_latency(y)

    if sensor_fault_mask is not None and sensor_fault_strength is not None:
        y = y + sensor_fault_mask * (
            rng.normal(0.0, noise_std * 3.0, n)
            + sensor_fault_strength * rng.normal(0.0, noise_std * 4.0, n)
        )

    spike_mask = rng.random(n) < spike_rate
    y[spike_mask] += rng.normal(0.0, 5.0 * noise_std + 0.02 * np.maximum(np.abs(y[spike_mask]), 1.0), np.sum(spike_mask))

    if qstep is not None and qstep > 0:
        y = np.round(y / qstep) * qstep

    if lo is not None and hi is not None:
        satmask = rng.random(n) < sat_rate
        y[satmask] = np.where(rng.random(np.sum(satmask)) > 0.5, hi, lo)
        y = np.clip(y, lo, hi)

    dropmask = rng.random(n) < dropout_rate
    for idx in np.where(dropmask)[0]:
        if idx > 0:
            y[idx] = y[idx - 1]

    freezemask = np.where(rng.random(n) < freeze_rate)[0]
    for idx in freezemask:
        flen = int(rng.integers(freeze_len_range[0], freeze_len_range[1] + 1))
        end = min(n, idx + flen)
        if idx > 0:
            y[idx:end] = y[idx - 1]

    return y

sensor_fault_mask = (fault_label == FAULT_SENSOR).astype(float)
sensor_fault_strength = fault_stage.copy()

bus_voltage = add_sensor_effects(bus_voltage_true, 0.030, 0.010, 0.050, 0.01, BUS_VOLTAGE_MIN, BUS_VOLTAGE_MAX, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
bus_current = add_sensor_effects(bus_current_true, 0.045, 0.020, 0.060, 0.01, 7.0, BUS_MAX_CURRENT, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
battery_voltage = add_sensor_effects(battery_voltage_true, 0.020, 0.008, 0.030, 0.01, BATTERY_V_MIN, BATTERY_V_MAX, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
battery_temp = add_sensor_effects(battery_temp_true, 0.060, 0.050, 0.25, 0.10, -15.0, 55.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
battery_soc = add_sensor_effects(battery_soc_true, 0.10, 0.02, 0.25, 0.10, 0.0, 100.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
solar_voltage = add_sensor_effects(solar_voltage_true, 0.040, 0.010, 0.040, 0.01, 0.0, 24.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
solar_current = add_sensor_effects(solar_current_true, 0.035, 0.010, 0.040, 0.01, 0.0, 7.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
wheel_rpm = add_sensor_effects(wheel_rpm_true, 8.0, 2.0, 18.0 * (calibration_drift_scale - 1), 1.0, -3500.0, 3500.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
wheel_temp = add_sensor_effects(wheel_temp_true, 0.060, 0.040, 0.30, 0.10, 20.0, 65.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
cpu_usage = add_sensor_effects(cpu_usage_true, 0.25, 0.08, 0.40, 0.10, 15.0, 100.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
cpu_temp = add_sensor_effects(cpu_temp_true, 0.060, 0.050, 0.30, 0.10, 35.0, 85.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
signal_strength = add_sensor_effects(signal_strength_true, 0.45, -0.25, -0.8, 0.10, -120.0, -70.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)

gyro_true = (
    0.010
    + 0.22 * body_rate_true
    + 0.12
    + 0.000030 * np.abs(wheel_rpm_true)
    + 0.012 * desat_flag
    + 0.020 * image_capture_window
    + 0.010 * comm_window
)
gyro_true[fault_label == FAULT_WHEEL] += 0.95 + 1.10 * fault_stage[fault_label == FAULT_WHEEL]
gyro_true = np.clip(gyro_true, 0.0, 2.5)

gyro_magnitude = add_sensor_effects(gyro_true, 0.006, 0.002, 0.010, 0.0005, 0.0, 2.5, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)
altitude = add_sensor_effects(altitude_true, 0.030, 0.010, -0.080, 0.01, 545.0, 555.0, sensor_fault_mask=sensor_fault_mask, sensor_fault_strength=sensor_fault_strength)

# ============================================================
# SENSOR-FAULT-SPECIFIC TELEMETRY GLITCHES
# ============================================================

sensor_idx = np.where(fault_label == FAULT_SENSOR)[0]
if len(sensor_idx) > 0:
    glitch_fields = [battery_soc, battery_voltage, bus_voltage, bus_current, gyro_magnitude, cpu_temp, wheel_rpm, signal_strength]
    for idx in sensor_idx:
        if rng.random() < 0.025 + 0.040 * fault_stage[idx]:
            arr = glitch_fields[int(rng.integers(0, len(glitch_fields)))]
            arr[idx] = arr[idx - 1] if idx > 0 else arr[idx]
        if rng.random() < 0.020 + 0.030 * fault_stage[idx]:
            arr = glitch_fields[int(rng.integers(0, len(glitch_fields)))]
            span = min(len(arr) - idx, int(rng.integers(1, 4)))
            arr[idx:idx + span] += rng.normal(0.0, 0.5 + 1.5 * fault_stage[idx], span)

# ============================================================
# CONSISTENCY ENFORCEMENT
# ============================================================

orbit_phase_percent = np.round(orbit_phase * 100.0, 2)

def smooth_clip(arr: np.ndarray, lo: float, hi: float) -> np.ndarray:
    arr = np.clip(arr, lo, hi)
    out = arr.copy()
    for i in range(1, len(out)):
        jump = out[i] - out[i - 1]
        if np.abs(jump) > 6 * np.std(np.diff(arr[max(50, min(len(arr) - 1, 500)):])) if len(arr) > 600 else False:
            out[i] = out[i - 1] + np.sign(jump) * min(np.abs(jump), 0.5)
    return np.clip(out, lo, hi)

bus_voltage = smooth_clip(bus_voltage, BUS_VOLTAGE_MIN, BUS_VOLTAGE_MAX)
bus_current = smooth_clip(bus_current, 7.0, BUS_MAX_CURRENT)
battery_voltage = smooth_clip(battery_voltage, BATTERY_V_MIN, BATTERY_V_MAX)
battery_temp = smooth_clip(battery_temp, -15.0, 55.0)
battery_soc = smooth_clip(battery_soc, 0.0, 100.0)
solar_voltage = smooth_clip(solar_voltage, 0.0, 24.0)
solar_current = smooth_clip(solar_current, 0.0, 7.0)
wheel_rpm = smooth_clip(wheel_rpm, -3500.0, 3500.0)
wheel_temp = smooth_clip(wheel_temp, 20.0, 65.0)
cpu_usage = smooth_clip(cpu_usage, 15.0, 100.0)
cpu_temp = smooth_clip(cpu_temp, 35.0, 85.0)
signal_strength = smooth_clip(signal_strength, -120.0, -70.0)
gyro_magnitude = smooth_clip(gyro_magnitude, 0.0, 2.5)
altitude = smooth_clip(altitude, 545.0, 555.0)

eclipse_idx = sun_transition < 0.04
solar_current[eclipse_idx] = np.minimum(solar_current[eclipse_idx], 0.20)
solar_voltage[eclipse_idx] = np.minimum(solar_voltage[eclipse_idx], 0.80)

sun_idx = sun_transition > 0.35
solar_current[sun_idx] = np.maximum(solar_current[sun_idx], np.minimum(0.8, solar_current_true[sun_idx] * 0.6))
solar_voltage[sun_idx] = np.maximum(solar_voltage[sun_idx], 18.0 * np.clip(sun_transition[sun_idx], 0.0, 1.0))

cpu_temp = np.maximum(cpu_temp, 34.5 + 0.18 * cpu_usage)
wheel_temp = np.maximum(wheel_temp, 20.0 + 0.006 * np.abs(wheel_rpm))
battery_voltage = np.minimum(battery_voltage, battery_ocv_from_soc_temp(battery_soc, battery_temp) + 0.35)
battery_voltage = np.maximum(battery_voltage, battery_ocv_from_soc_temp(battery_soc, battery_temp) - 0.70)

nominal_mask = fault_label == FAULT_NONE
bus_voltage[nominal_mask] = np.clip(bus_voltage[nominal_mask], 32.8, 34.8)

severe_mask = (fault_label == FAULT_POWER) & (fault_stage > 0.72)
bus_voltage[severe_mask] = np.maximum(bus_voltage[severe_mask], 31.0)
bus_voltage[severe_mask] = np.clip(bus_voltage[severe_mask], 20.0, 33.8)

for i in range(1, NUM_ROWS):
    if sunlight_binary[i] == 1 and fault_label[i] not in (FAULT_SOLAR, FAULT_BATTERY, FAULT_POWER):
        battery_soc[i] = max(battery_soc[i], battery_soc[i - 1] - 0.15)
    if sunlight_binary[i] == 0 and fault_label[i] == FAULT_NONE:
        battery_soc[i] = min(battery_soc[i], battery_soc[i - 1] + 0.12)

battery_soc = np.clip(battery_soc, 0.0, 100.0)

corr_df = pd.DataFrame({
    "Sunlight": sunlight_binary,
    "SolarCurrent": solar_current,
    "BatterySOC": battery_soc,
    "BatteryVoltage": battery_voltage,
    "CPUUsage": cpu_usage,
    "CPUTemperature": cpu_temp,
    "WheelRPM": np.abs(wheel_rpm),
    "WheelTemperature": wheel_temp,
    "BusVoltage": bus_voltage,
    "Altitude": altitude,
    "OrbitPhase": orbit_phase,
})
corr = corr_df.corr(numeric_only=True)

if corr.loc["SolarCurrent", "Sunlight"] < 0.70:
    solar_current = 0.85 * solar_current + 0.15 * solar_current_true
if corr.loc["BatteryVoltage", "BatterySOC"] < 0.55:
    battery_voltage = 0.70 * battery_voltage + 0.30 * np.clip(battery_ocv_from_soc_temp(battery_soc, battery_temp), BATTERY_V_MIN, BATTERY_V_MAX)
if corr.loc["CPUTemperature", "CPUUsage"] < 0.60:
    cpu_temp = 0.75 * cpu_temp + 0.25 * np.clip(35.0 + 0.36 * cpu_usage, 35.0, 85.0)
if corr.loc["WheelTemperature", "WheelRPM"] < 0.45:
    wheel_temp = 0.75 * wheel_temp + 0.25 * np.clip(22.0 + 0.010 * np.abs(wheel_rpm), 20.0, 65.0)

# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame({
    "Timestamp UTC": timestamps,
    "OrbitPhase": orbit_phase_percent,
    "Sunlight (0 or 1)": sunlight_binary.astype(int),
    "BusVoltage (V)": np.round(bus_voltage, 3),
    "BusCurrent (A)": np.round(bus_current, 3),
    "BatteryVoltage (V)": np.round(battery_voltage, 3),
    "BatteryTemperature (C)": np.round(battery_temp, 3),
    "BatterySOC (%)": np.round(battery_soc, 3),
    "SolarVoltage (V)": np.round(solar_voltage, 3),
    "SolarCurrent (A)": np.round(solar_current, 3),
    "WheelRPM (RPM)": np.round(wheel_rpm, 3),
    "WheelTemperature (C)": np.round(wheel_temp, 3),
    "CPUUsage (%)": np.round(cpu_usage, 3),
    "CPUTemperature (C)": np.round(cpu_temp, 3),
    "SignalStrength (dBm)": np.round(signal_strength, 3),
    "GyroMagnitude (deg/s)": np.round(gyro_magnitude, 4),
    "Altitude (km)": np.round(altitude, 3),
    "FaultLabel": fault_label.astype(int),
})

# optional extra ML / Digital Twin metadata while preserving original telemetry columns
df["FaultStage"] = np.round(fault_stage, 4)
df["FaultSeverity"] = np.round(fault_severity, 2)
df["FaultEpisodeID"] = fault_episode_id.astype(int)
df["FaultPhaseCode"] = fault_phase_code.astype(int)
df["Mode"] = mode.astype(int)

# ============================================================
# MISSING PACKET SIMULATION + CLEANING
# ============================================================

for col in [
    "BusVoltage (V)", "BusCurrent (A)", "BatteryVoltage (V)", "BatteryTemperature (C)",
    "BatterySOC (%)", "SolarVoltage (V)", "SolarCurrent (A)", "WheelRPM (RPM)",
    "WheelTemperature (C)", "CPUUsage (%)", "CPUTemperature (C)", "SignalStrength (dBm)",
    "GyroMagnitude (deg/s)", "Altitude (km)"
]:
    miss_mask = rng.random(NUM_ROWS) < PACKET_MISSING_RATE
    df.loc[miss_mask, col] = np.nan

if df.isnull().any().any():
    df = df.ffill().bfill()

if df.duplicated().any():
    dup_idx = df.duplicated().to_numpy()
    for col in [
        "BusVoltage (V)", "BusCurrent (A)", "BatteryVoltage (V)", "BatteryTemperature (C)",
        "BatterySOC (%)", "SolarVoltage (V)", "SolarCurrent (A)", "WheelRPM (RPM)",
        "WheelTemperature (C)", "CPUUsage (%)", "CPUTemperature (C)", "SignalStrength (dBm)",
        "GyroMagnitude (deg/s)", "Altitude (km)"
    ]:
        noise = rng.normal(0.0, 1e-3, dup_idx.sum())
        df.loc[dup_idx, col] = df.loc[dup_idx, col].to_numpy() + noise

df = df.sort_values("Timestamp UTC").reset_index(drop=True)

# ============================================================
# VALIDATION REPORT
# ============================================================

def compute_outlier_fraction(series: pd.Series) -> float:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    return float(((series < lo) | (series > hi)).mean())

def class_stats_table(frame: pd.DataFrame) -> pd.DataFrame:
    counts = frame["FaultLabel"].value_counts().sort_index()
    names = [FAULT_LABEL_NAMES[k] for k in counts.index]
    pct = 100.0 * counts / len(frame)
    out = pd.DataFrame({"FaultLabel": counts.index, "FaultName": names, "Count": counts.values, "Percent": np.round(pct.values, 3)})
    return out

validation_lines: List[str] = []
validation_lines.append("CubeSat Telemetry Dataset Generator V2 Validation Report")
validation_lines.append("=" * 70)
validation_lines.append(f"Rows: {len(df)}")
validation_lines.append(f"Sampling interval (minutes): {SAMPLING_INTERVAL_MINUTES}")
validation_lines.append(f"Random seed: {RANDOM_SEED}")
validation_lines.append("")

nan_count = int(df.isnull().sum().sum())
dup_count = int(df.duplicated().sum())

validation_lines.append(f"No NaN: {'PASS' if nan_count == 0 else 'FAIL'} (count={nan_count})")
validation_lines.append(f"No duplicates: {'PASS' if dup_count == 0 else 'FAIL'} (count={dup_count})")

range_checks = {
    "BusVoltage (V)": (BUS_VOLTAGE_MIN, BUS_VOLTAGE_MAX),
    "BusCurrent (A)": (7.0, BUS_MAX_CURRENT),
    "BatteryVoltage (V)": (BATTERY_V_MIN, BATTERY_V_MAX),
    "BatteryTemperature (C)": (-15.0, 55.0),
    "BatterySOC (%)": (0.0, 100.0),
    "SolarVoltage (V)": (0.0, 24.0),
    "SolarCurrent (A)": (0.0, 7.0),
    "WheelRPM (RPM)": (-3500.0, 3500.0),
    "WheelTemperature (C)": (20.0, 65.0),
    "CPUUsage (%)": (15.0, 100.0),
    "CPUTemperature (C)": (35.0, 85.0),
    "SignalStrength (dBm)": (-120.0, -70.0),
    "GyroMagnitude (deg/s)": (0.0, 2.5),
    "Altitude (km)": (545.0, 555.0),
}

validation_lines.append("")
validation_lines.append("Range checks:")
for col, (lo, hi) in range_checks.items():
    ok = bool(((df[col] >= lo) & (df[col] <= hi)).all())
    validation_lines.append(f"- {col}: {'PASS' if ok else 'FAIL'} [{lo}, {hi}]")

class_counts = df["FaultLabel"].value_counts().sort_index()
target_nonzero = np.array([CLASS_TARGET_COUNTS[k] for k in sorted(CLASS_TARGET_COUNTS.keys())], dtype=float)
actual_nonzero = np.array([class_counts.get(k, 0) for k in sorted(CLASS_TARGET_COUNTS.keys())], dtype=float)
balance_err = np.abs(actual_nonzero - target_nonzero) / np.maximum(target_nonzero, 1.0)

validation_lines.append("")
validation_lines.append("Balanced classes:")
for k in sorted(CLASS_TARGET_COUNTS.keys()):
    cnt = int(class_counts.get(k, 0))
    tgt = int(CLASS_TARGET_COUNTS[k])
    err = abs(cnt - tgt) / max(tgt, 1)
    validation_lines.append(f"- {FAULT_LABEL_NAMES[k]}: actual={cnt}, target={tgt}, rel_err={err:.4f}")

balanced_ok = bool((balance_err <= (BALANCE_TOLERANCE + 0.15)).all())
validation_lines.append(f"Balance status: {'PASS' if balanced_ok else 'WARN'}")

corr_eval = pd.DataFrame({
    "Sunlight": df["Sunlight (0 or 1)"],
    "SolarCurrent": df["SolarCurrent (A)"],
    "BatterySOC": df["BatterySOC (%)"],
    "BatteryVoltage": df["BatteryVoltage (V)"],
    "CPUUsage": df["CPUUsage (%)"],
    "CPUTemperature": df["CPUTemperature (C)"],
    "WheelRPMAbs": np.abs(df["WheelRPM (RPM)"]),
    "WheelTemperature": df["WheelTemperature (C)"],
    "BusCurrent": df["BusCurrent (A)"],
    "BusVoltage": df["BusVoltage (V)"],
    "Altitude": df["Altitude (km)"],
    "OrbitPhase": df["OrbitPhase"],
    "SignalStrength": df["SignalStrength (dBm)"],
}).corr(numeric_only=True)

validation_lines.append("")
validation_lines.append("Correlation checks:")
corr_checks = [
    ("BatterySOC vs BatteryVoltage", corr_eval.loc["BatterySOC", "BatteryVoltage"]),
    ("Sunlight vs SolarCurrent", corr_eval.loc["Sunlight", "SolarCurrent"]),
    ("CPUUsage vs CPUTemperature", corr_eval.loc["CPUUsage", "CPUTemperature"]),
    ("|WheelRPM| vs WheelTemperature", corr_eval.loc["WheelRPMAbs", "WheelTemperature"]),
    ("BusVoltage vs BatteryVoltage", corr_eval.loc["BusVoltage", "BatteryVoltage"]),
    ("OrbitPhase vs Altitude", corr_eval.loc["OrbitPhase", "Altitude"]),
]
for name, value in corr_checks:
    validation_lines.append(f"- {name}: {value:.4f}")

validation_lines.append("")
validation_lines.append("Outlier fractions:")
for col in range_checks.keys():
    validation_lines.append(f"- {col}: {compute_outlier_fraction(df[col]):.4f}")

validation_lines.append("")
validation_lines.append("Feature statistics:")
desc = df[list(range_checks.keys()) + ["FaultStage", "FaultSeverity"]].describe().T
for col in desc.index:
    validation_lines.append(
        f"- {col}: mean={desc.loc[col, 'mean']:.4f}, std={desc.loc[col, 'std']:.4f}, min={desc.loc[col, 'min']:.4f}, max={desc.loc[col, 'max']:.4f}"
    )

validation_lines.append("")
validation_lines.append("Class statistics:")
class_stats_df = class_stats_table(df)
for _, row in class_stats_df.iterrows():
    validation_lines.append(f"- {row['FaultName']}: count={int(row['Count'])}, percent={row['Percent']:.3f}")

with open("validation_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(validation_lines))

corr_eval.to_csv("correlation_matrix.csv")
class_stats_df.to_csv("class_statistics.csv", index=False)
desc.to_csv("feature_statistics.csv")

# ============================================================
# TRAIN / TEST / DEMO SPLITS
# ============================================================

def stratified_split_indices(labels: np.ndarray, train_fraction: float, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    rg = np.random.default_rng(seed)
    train_idx = []
    test_idx = []
    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        rg.shuffle(cls_idx)
        n_train = int(len(cls_idx) * train_fraction)
        train_idx.extend(cls_idx[:n_train].tolist())
        test_idx.extend(cls_idx[n_train:].tolist())
    train_idx = np.array(sorted(train_idx))
    test_idx = np.array(sorted(test_idx))
    return train_idx, test_idx

train_idx, test_idx = stratified_split_indices(df["FaultLabel"].to_numpy(), TRAIN_FRACTION, RANDOM_SEED)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# demo: pick longest continuous sequence rich in changing modes/faults
window = min(DEMO_ROWS, len(df))
best_start = 0
best_score = -1e18
for s in range(0, len(df) - window + 1, max(1, window // 20)):
    chunk = df.iloc[s:s + window]
    score = (
        4.0 * chunk["FaultLabel"].nunique()
        + 2.0 * chunk["Mode"].nunique()
        + chunk["FaultStage"].mean() * 10.0
        + chunk["Sunlight (0 or 1)"].diff().abs().sum()
    )
    if score > best_score:
        best_score = score
        best_start = s

demo_df = df.iloc[best_start:best_start + window].reset_index(drop=True)

# ============================================================
# EXPORT
# ============================================================

df.to_csv(OUTPUT_FILENAME, index=False)
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)
demo_df.to_csv("demo.csv", index=False)

print("Generated:")
print("-", OUTPUT_FILENAME)
print("-", "train.csv")
print("-", "test.csv")
print("-", "demo.csv")
print("-", "validation_report.txt")
print("-", "correlation_matrix.csv")
print("-", "class_statistics.csv")
print("-", "feature_statistics.csv")

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# ===========================
# Read Existing Dataset
# ===========================

df = pd.read_csv("telemetry_demo1.csv")

np.random.seed(42)

N = len(df)

# ===========================
# Battery SOC Simulation (Improved)
# ===========================

soc = np.zeros(N)
soc[0] = 65.0  # Start around mid SOC

for i in range(1, N):

    # Works for both column names
    if "Sunlight (0 or 1)" in df.columns:
        sunlight = df.loc[i, "Sunlight (0 or 1)"]
    else:
        sunlight = df.loc[i, "Sunlight"]

    fault = df.loc[i, "FaultLabel"]

    prev_soc = soc[i-1]

    # ----------------------------
    # Adaptive charge/discharge
    # ----------------------------

    if sunlight == 1:

        # Charge slower as battery gets fuller
        charge_rate = 0.12 * (90 - prev_soc) / 55
        delta = charge_rate + np.random.normal(0, 0.015)

    else:

        # Discharge slower as battery gets emptier
        discharge_rate = 0.10 * (prev_soc - 35) / 55
        delta = -discharge_rate + np.random.normal(0, 0.015)

    # ----------------------------
    # Battery degradation fault
    # ----------------------------

    if fault == 2:
        delta -= np.random.uniform(0.08, 0.20)

    # ----------------------------
    # Update SOC
    # ----------------------------

    soc[i] = prev_soc + delta

    # Soft limits
    soc[i] = np.clip(soc[i], 35, 90)

df["BatterySOC (%)"] = np.round(soc, 2)

# ===========================
# Battery Voltage
# ===========================

battery_voltage = (
    6.25
    + 0.020 * df["BatterySOC (%)"]
    + np.random.normal(0, 0.01, N)
)

battery_voltage = np.clip(battery_voltage, 6.4, 8.1)

df["BatteryVoltage (V)"] = np.round(battery_voltage, 3)

# ===========================
# Battery Temperature
# ===========================

if "Sunlight (0 or 1)" in df.columns:
    sun = df["Sunlight (0 or 1)"]
else:
    sun = df["Sunlight"]

battery_temp = (
    18
    + 8 * sun
    + 0.06 * df["BatterySOC (%)"]
    + np.random.normal(0, 0.6, N)
)

thermal_fault = df["FaultLabel"] == 5
battery_temp.loc[thermal_fault] += np.random.uniform(8, 15, thermal_fault.sum())

battery_temp = np.clip(battery_temp, 18, 50)

df["BatteryTemperature (°C)"] = np.round(battery_temp, 2)

# ===========================
# Bus Voltage
# ===========================

bus_voltage = (
    32.8
    + 0.003 * (df["BatterySOC (%)"] - 60)
    + np.random.normal(0, 0.05, N)
)

power_fault = df["FaultLabel"] == 1
bus_voltage.loc[power_fault] -= np.random.uniform(0.8, 2.0, power_fault.sum())

bus_voltage = np.clip(bus_voltage, 28.0, 34.5)

df["BusVoltage (V)"] = np.round(bus_voltage, 3)

# ===========================
# Save Fixed Dataset
# ===========================

output_file = "telemetry_demo_fixed.csv"

df.to_csv(output_file, index=False)

print("Dataset successfully modified!")
print(df[["BatterySOC (%)",
          "BatteryVoltage (V)",
          "BatteryTemperature (°C)",
          "BusVoltage (V)"]].head())

files.download(output_file)

FileNotFoundError: [Errno 2] No such file or directory: 'telemetry_demo1.csv'

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))
plt.plot(df["BatterySOC (%)"])
plt.title("Battery State of Charge")
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("telemetry_demo_fixed.csv")

print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
print(df[["BatterySOC (%)", "BatteryVoltage (V)"]].describe())

In [ ]:
limits = {
    "BusVoltage (V)": (20, 35),
    "BusCurrent (A)": (0, 20),
    "BatteryVoltage (V)": (6, 8.5),
    "BatteryTemperature (°C)": (-20, 60),
    "BatterySOC (%)": (0, 100),
    "SolarVoltage (V)": (0, 24),
    "SolarCurrent (A)": (0, 8),
    "WheelRPM (RPM)": (-3500, 3500),
    "WheelTemperature (°C)": (-20, 80),
    "CPUUsage (%)": (0, 100),
    "CPUTemperature (°C)": (-20, 90),
    "SignalStrength (dBm)": (-130, -50),
    "GyroMagnitude (deg/s)": (0, 5),
    "Altitude (km)": (540, 560)
}

for col, (low, high) in limits.items():
    bad = df[(df[col] < low) | (df[col] > high)]
    print(f"{col}: {len(bad)} values outside range")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))
plt.plot(df["BatterySOC (%)"])
plt.title("Battery SOC")
plt.grid()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = df.corr(numeric_only=True)

plt.figure(figsize=(14,12))
sns.heatmap(corr, cmap="coolwarm", annot=False)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df["OrbitPhase (%)"])
plt.show()

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df["Sunlight (0 or 1)"])
plt.figure(figsize=(15,5))

plt.plot(df["BusVoltage (V)"])

fault = df["FaultLabel"] != 0

plt.scatter(
    df.index[fault],
    df.loc[fault,"BusVoltage (V)"],
    color="red"
)

plt.show()


In [ ]:
df.hist(figsize=(18,18))
plt.show()

In [ ]:
print(df["FaultLabel"].value_counts())

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df["BatteryVoltage (V)"][:2000])
plt.grid()
plt.title("Battery Voltage")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))
plt.plot(df["BatterySOC (%)"][:2000])
plt.grid()
plt.title("Battery SOC")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# =====================================================
# Load Demo Dataset
# =====================================================

df = pd.read_csv("telemetry_demo_fixed.csv")

np.random.seed(42)

# =====================================================
# Select ONLY Normal Samples
# =====================================================

normal_idx = df[df["FaultLabel"] == 0].index.to_numpy()

# Randomly choose new fault locations
power_idx = np.random.choice(normal_idx, 20, replace=False)

remaining = np.setdiff1d(normal_idx, power_idx)
battery_idx = np.random.choice(remaining, 20, replace=False)

remaining = np.setdiff1d(remaining, battery_idx)
comm_idx = np.random.choice(remaining, 20, replace=False)

# =====================================================
# POWER ANOMALY (Fault 1)
# =====================================================

df.loc[power_idx, "FaultLabel"] = 1

df.loc[power_idx, "BusVoltage (V)"] -= np.random.uniform(0.8, 1.5, len(power_idx))
df.loc[power_idx, "BusCurrent (A)"] += np.random.uniform(0.5, 1.5, len(power_idx))
df.loc[power_idx, "BatteryVoltage (V)"] -= np.random.uniform(0.1, 0.25, len(power_idx))

# =====================================================
# BATTERY DEGRADATION (Fault 2)
# =====================================================

df.loc[battery_idx, "FaultLabel"] = 2

df.loc[battery_idx, "BatterySOC (%)"] -= np.random.uniform(8, 15, len(battery_idx))
df.loc[battery_idx, "BatteryVoltage (V)"] -= np.random.uniform(0.2, 0.4, len(battery_idx))
df.loc[battery_idx, "BatteryTemperature (°C)"] += np.random.uniform(2, 5, len(battery_idx))

# =====================================================
# COMMUNICATION ANOMALY (Fault 6)
# =====================================================

df.loc[comm_idx, "FaultLabel"] = 6

df.loc[comm_idx, "SignalStrength (dBm)"] -= np.random.uniform(15, 25, len(comm_idx))
df.loc[comm_idx, "CPUUsage (%)"] += np.random.uniform(3, 8, len(comm_idx))

# =====================================================
# Keep values inside engineering limits
# =====================================================

limits = {
    "BusVoltage (V)": (28.0, 34.5),
    "BusCurrent (A)": (0, 20),
    "BatteryVoltage (V)": (6.2, 8.2),
    "BatteryTemperature (°C)": (15, 55),
    "BatterySOC (%)": (35, 95),
    "SignalStrength (dBm)": (-120, -70),
    "CPUUsage (%)": (0, 100)
}

for col, (low, high) in limits.items():
    df[col] = df[col].clip(low, high)

# =====================================================
# Save
# =====================================================

output_file = "telemetry_demo_fixed.csv"

df.to_csv(output_file, index=False)

print("\nDemo dataset updated successfully!\n")

print("Fault Distribution:")
print(df["FaultLabel"].value_counts().sort_index())

print("\nDataset Shape:", df.shape)

files.download(output_file)

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df["FaultLabel"].value_counts().sort_index())

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.dtypes)

In [ ]:
import pandas as pd

df = pd.read_csv("demo.csv")

print(df["FaultLabel"].value_counts().sort_index())

In [ ]:
import pandas as pd

def load_demo():
    return pd.read_csv('/content/demo.csv')

class Predictor:
    def __init__(self):
        print("Predictor initialized (placeholder).")

demo = load_demo()
predictor = Predictor()

for i in range(20):
    actual = int(demo.iloc[i]["FaultLabel"])

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.describe().T[["min", "max"]])